In [1]:
import os
import cv2
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

import torchvision.models as models

from sklearn.model_selection import train_test_split

import albumentations as A
from albumentations.pytorch import ToTensorV2


def locate_path(*candidates):
    """
    Return the first existing path from the candidate list.
    Adds common Kaggle prefixes to improve robustness.
    """
    extra_prefixes = [
        "/kaggle/input",
        "/kaggle/working",
        "/kaggle/working/cassava-leaf-disease-classification",
        "/kaggle/input/cassava-leaf-disease-classification",
    ]
    for p in candidates:
        if Path(p).exists():
            return p
        for prefix in extra_prefixes:
            candidate = Path(prefix) / p
            if candidate.exists():
                return str(candidate)
    raise FileNotFoundError(f"None of the candidate paths exist: {candidates}")


train_csv_path = locate_path(
    "data/cassava-leaf-disease-classification/train.csv",
    "input/cassava-leaf-disease-classification/train.csv",
    "train.csv",
    "cassava-leaf-disease-classification/train.csv",
)
test_csv_path = locate_path(
    "data/cassava-leaf-disease-classification/sample_submission.csv",
    "input/cassava-leaf-disease-classification/sample_submission.csv",
    "sample_submission.csv",
    "cassava-leaf-disease-classification/sample_submission.csv",
)

train_img_dir = locate_path(
    "data/cassava-leaf-disease-classification/train_images",
    "input/cassava-leaf-disease-classification/train_images",
    "train_images",
    "cassava-leaf-disease-classification/train_images",
)
test_img_dir = locate_path(
    "data/cassava-leaf-disease-classification/test_images",
    "input/cassava-leaf-disease-classification/test_images",
    "test_images",
    "cassava-leaf-disease-classification/test_images",
)



In [2]:
train_df = pd.read_csv(train_csv_path)
test_df = pd.read_csv(test_csv_path)



In [3]:
train_transform = A.Compose(
    [
        A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=1.0),
        A.Resize(384, 384, p=1.0),
        A.HorizontalFlip(p=0.5),
        A.ShiftScaleRotate(0.1, 0.1, 20, p=0.7),
        A.HueSaturationValue(10, 15, 10, p=0.5),
        A.RandomBrightnessContrast(0.2, 0.2, p=0.5),
        A.GaussNoise((0.1, 0.5), p=0.4),
        A.Blur(blur_limit=3, p=0.3),
        A.CoarseDropout(
            max_holes=8,
            max_height=20,
            max_width=20,
            min_holes=1,
            min_height=10,
            min_width=10,
            fill_value=0,
            p=0.5,
        ),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ]
)

val_transform = A.Compose(
    [
        A.Resize(384, 384, p=1.0),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ]
)

tta_transform = A.Compose(
    [
        A.ShiftScaleRotate(0.1, 0.1, 20, p=0.7),
        A.HueSaturationValue(10, 15, 10, p=0.5),
        A.RandomBrightnessContrast(0.2, 0.2, p=0.5),
        A.HorizontalFlip(p=0.5),
        A.GaussNoise((0.1, 0.5), p=0.4),
        A.Blur(blur_limit=3, p=0.3),
        A.CoarseDropout(
            max_holes=8,
            max_height=20,
            max_width=20,
            min_holes=1,
            min_height=10,
            min_width=10,
            fill_value=0,
            p=0.5,
        ),
        A.Resize(384, 384, p=1.0),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ]
)




/usr/local/lib/python3.11/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/tmp/ipykernel_55/2879136161.py:11: UserWarning: Argument(s) 'max_holes, max_height, max_width, min_holes, min_height, min_width, fill_value' are not valid for transform CoarseDropout
  A.CoarseDropout(
/tmp/ipykernel_55/2879136161.py:42: UserWarning: Argument(s) 'max_holes, max_height, max_width, min_holes, min_height, min_width, fill_value' are not valid for transform CoarseDropout
  A.CoarseDropout(


In [4]:
class CassavaDataset(Dataset):
    """Pre‑load and resize all training images once to avoid per‑epoch disk I/O."""

    def __init__(self, df, img_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
        self.images = []
        for img_name in self.df["image_id"]:
            img_path = os.path.join(self.img_dir, img_name)
            img = cv2.imread(img_path)
            if img is None:
                img = np.zeros((384, 384, 3), dtype=np.uint8)
            else:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = cv2.resize(img, (384, 384), interpolation=cv2.INTER_LINEAR)
            self.images.append(img)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img = self.images[idx]
        label = int(self.df.iloc[idx]["label"])
        if self.transform:
            img = self.transform(image=img)["image"]
        return img, label


class CassavaTestDataset(Dataset):
    """Pre‑load all test images into memory once for fast TTA."""

    def __init__(self, df, img_dir):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.images = []
        for img_name in self.df["image_id"]:
            img_path = os.path.join(self.img_dir, img_name)
            img = cv2.imread(img_path)
            if img is None:
                img = np.zeros((384, 384, 3), dtype=np.uint8)
            else:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = cv2.resize(img, (384, 384), interpolation=cv2.INTER_LINEAR)
            self.images.append(img)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        return self.images[idx], self.df.iloc[idx]["image_id"]




In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")



In [6]:
train_split, val_split = train_test_split(
    train_df,
    test_size=0.1,
    stratify=train_df["label"],
    random_state=42,
)

train_dataset = CassavaDataset(train_split, train_img_dir, transform=train_transform)
val_dataset = CassavaDataset(val_split, train_img_dir, transform=val_transform)

num_workers = min(os.cpu_count() or 4, 4)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2,
)



In [7]:
model = models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.IMAGENET1K_V1)
if isinstance(model.classifier, nn.Sequential):
    num_features = model.classifier[1].in_features
else:
    num_features = model.classifier.in_features
model.classifier = nn.Sequential(nn.Dropout(p=0.5), nn.Linear(num_features, 5))
model = model.to(device)

try:
    model = torch.compile(model)
except Exception:
    pass

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)
scaler = GradScaler()




Downloading: "https://download.pytorch.org/models/efficientnet_v2_s-dd5fe13b.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_v2_s-dd5fe13b.pth
100%|██████████| 82.7M/82.7M [00:00<00:00, 108MB/s]
/tmp/ipykernel_55/513588456.py:16: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


In [8]:
def train_one_epoch(model, loader, optimizer, criterion, device, scaler):
    model.train()
    running_loss = 0.0
    for imgs, targets in loader:
        imgs, targets = imgs.to(device, non_blocking=True), targets.to(
            device, non_blocking=True
        )
        optimizer.zero_grad()
        with autocast():
            outputs = model(imgs)
            loss = criterion(outputs, targets)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item() * imgs.size(0)
    return running_loss / len(loader.dataset)


def evaluate(model, loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.inference_mode():
        for imgs, targets in loader:
            imgs, targets = imgs.to(device, non_blocking=True), targets.to(
                device, non_blocking=True
            )
            with autocast():
                outputs = model(imgs)
            _, preds = torch.max(outputs, 1)
            correct += (preds == targets).sum().item()
            total += targets.size(0)
    return correct / total




In [9]:
epochs = 3  # modest number to keep runtime reasonable while improving accuracy
for epoch in range(epochs):
    train_loss = train_one_epoch(
        model, train_loader, optimizer, criterion, device, scaler
    )
    val_acc = evaluate(model, val_loader, device)
    print(
        f"Epoch {epoch+1}/{epochs} - Train loss: {train_loss:.4f} - Val acc: {val_acc:.4f}"
    )




/tmp/ipykernel_55/2426965210.py:9: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_55/2426965210.py:9: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_55/2426965210.py:27: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_55/2426965210.py:27: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 1/3 - Train loss: 0.7508 - Val acc: 0.8655


/tmp/ipykernel_55/2426965210.py:9: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 2/3 - Train loss: 0.4890 - Val acc: 0.8783
Epoch 3/3 - Train loss: 0.4240 - Val acc: 0.8809


In [10]:
def tta_predict_batch_batch(model, images_np, tta_transform, device, n_tta=5):
    """
    images_np: (B, H, W, C) uint8
    Returns mean probability tensor of shape (B, 5)
    """
    model.eval()
    B = images_np.shape[0]
    probs_sum = torch.zeros((B, 5), device=device, dtype=torch.float32)
    with torch.inference_mode():
        for _ in range(n_tta):
            aug_batch = [tta_transform(image=images_np[i])["image"] for i in range(B)]
            batch_tensor = torch.stack(aug_batch).to(device, non_blocking=True)
            with autocast():
                out = model(batch_tensor)
            probs = F.softmax(out, dim=1)
            probs_sum += probs
    return probs_sum / n_tta




In [11]:
def test_collate_fn(batch):
    """Return a list of images and a list of ids without converting to tensors."""
    imgs, ids = zip(*batch)
    return list(imgs), list(ids)


test_dataset = CassavaTestDataset(test_df, test_img_dir)
test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2,
    collate_fn=test_collate_fn,
)



In [12]:
ensemble_predictions, image_names = [], []
model.eval()
with torch.no_grad():
    for batch_imgs, batch_names in test_loader:
        batch_np = np.stack(batch_imgs)  # (B, H, W, C) uint8
        probs = tta_predict_batch_batch(model, batch_np, tta_transform, device, n_tta=5)
        preds = probs.argmax(dim=1).cpu().tolist()
        ensemble_predictions.extend(preds)
        image_names.extend(batch_names)



/tmp/ipykernel_55/635749196.py:13: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


In [13]:
submission_df = pd.DataFrame({"image_id": image_names, "label": ensemble_predictions})
submission_path = "submission.csv"
submission_df.to_csv(submission_path, index=False)
print(f"Submission file saved as '{submission_path}' with {len(submission_df)} rows.")

Submission file saved as 'submission.csv' with 2676 rows.
